In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(
    "../data/spy_modeling_data.csv",
    index_col="Date",
    parse_dates=True
)

df.shape

(8428, 16)

In [3]:
df.head()

,return_1d,return_3d,return_5d,return_10d,return_20d,price_vs_sma5,price_vs_sma20,sma5_vs_sma20,volatility_5d,volatility_20d,volatility_ratio,high_low_range,overnight_gap,intraday_return,volume_ratio,target_3d
Date,,,,,,,,,,,,,,,,
1993-03-01,-0.002814,0.000706,0.012866,-0.007008,0.007824,0.001980,-0.000212,-0.002187,0.006073,0.007807,0.777961,0.007763,0.003519,-0.006311,0.242519,1.0
1993-03-02,0.014820,0.013389,0.028612,0.033789,0.015536,0.011109,0.013818,0.002679,0.007720,0.008328,0.927048,0.015299,0.000706,0.014104,0.703432,0.0
1993-03-03,0.004173,0.016186,0.019774,0.038849,0.017618,0.011346,0.017152,0.005741,0.006591,0.008357,0.788662,0.004848,0.001391,0.002778,1.064048,1.0
1993-03-04,-0.005540,0.013408,0.011980,0.033837,0.001394,0.003354,0.011445,0.008065,0.007884,0.008151,0.967249,0.006964,0.001385,-0.006916,0.370992,1.0
1993-03-05,-0.002786,-0.004172,0.007741,0.027259,-0.005556,-0.000977,0.008912,0.009899,0.008232,0.008116,1.014345,0.009078,0.001393,-0.004172,0.184613,1.0


In [4]:
df.tail()

,return_1d,return_3d,return_5d,return_10d,return_20d,price_vs_sma5,price_vs_sma20,sma5_vs_sma20,volatility_5d,volatility_20d,volatility_ratio,high_low_range,overnight_gap,intraday_return,volume_ratio,target_3d
Date,,,,,,,,,,,,,,,,
2026-08-17,-0.004727,0.000233,-0.000466,0.019798,0.041208,-0.001703,0.019723,0.021462,0.004780,0.008519,0.561133,0.005526,-0.000206,-0.004522,0.744923,0.0
2026-08-18,-0.006756,-0.013408,-0.004036,-0.005030,0.025619,-0.007650,0.011554,0.019352,0.005561,0.008601,0.646550,0.003362,-0.005138,-0.001626,0.942491,0.0
2026-08-19,0.002098,-0.009377,-0.004440,-0.000948,0.028967,-0.004685,0.012232,0.016997,0.005503,0.008583,0.641205,0.005682,0.003792,-0.001688,0.858043,0.0
2026-08-20,-0.008400,-0.013033,-0.019643,-0.007755,0.033081,-0.009127,0.002119,0.011349,0.004145,0.008289,0.500133,0.008012,-0.004031,-0.004387,0.979384,0.0
2026-08-21,0.004091,-0.002254,-0.013680,-0.009751,0.036255,-0.002319,0.004451,0.006786,0.005527,0.008304,0.665520,0.004806,0.004524,-0.000431,0.848262,0.0


In [5]:
feature_cols = [
    col for col in df.columns
    if col != "target_3d"
]

X = df[feature_cols].copy()
y = df["target_3d"].astype(int).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Number of features:", len(feature_cols))

X shape: (8428, 15)
y shape: (8428,)
Number of features: 15


In [6]:
HORIZON = 3

train_df = df.loc[:"2018-12-31"].copy()
val_df = df.loc["2019-01-01":"2022-12-31"].copy()
test_df = df.loc["2023-01-01":].copy()

#Purging observations whose targets cross boundaries
train_df = train_df.iloc[:-HORIZON].copy()
val_df = val_df.iloc[:-HORIZON].copy()


In [7]:
for name, split_df in [
    ("TRAIN", train_df),
    ("VALIDATION",val_df),
    ("TEST", test_df)
]:
    print(f"\n{name}")
    print("Rows:",len(split_df))
    print("Start:", split_df.index.min())
    print("End: ",split_df.index.max())
    print("Positive target rate:", split_df["target_3d"].mean())


TRAIN
Rows: 6505
Start: 1993-03-01 00:00:00
End:  2018-12-26 00:00:00
Positive target rate: 0.41275941583397385

VALIDATION
Rows: 1005
Start: 2019-01-02 00:00:00
End:  2022-12-27 00:00:00
Positive target rate: 0.46567164179104475

TEST
Rows: 912
Start: 2023-01-03 00:00:00
End:  2026-08-21 00:00:00
Positive target rate: 0.4550438596491228


In [8]:
X_train = train_df[feature_cols].copy()
y_train = train_df["target_3d"].astype(int).copy()

X_val = val_df[feature_cols].copy()
y_val = val_df["target_3d"].astype(int).copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target_3d"].astype(int).copy()

print("TRAIN:", X_train.shape, y_train.shape)
print("VAL:  ", X_val.shape, y_val.shape)
print("TEST: ", X_test.shape, y_test.shape)

TRAIN: (6505, 15) (6505,)
VAL:   (1005, 15) (1005,)
TEST:  (912, 15) (912,)


In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

In [10]:
dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train,y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.59,0.41]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](15,)","['return_1d','return_3d','return_5d',...,'overnight_gap','intraday_return', 'volume_ratio']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,15
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [11]:
dummy_val_pred = dummy.predict(X_val)

print("First 20 predictions:")
print(dummy_val_pred[:20])

print("\nUnique predictions:")
print(np.unique(dummy_val_pred))

First 20 predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Unique predictions:
[0]


In [12]:
dummy_val_accuracy = accuracy_score(
    y_val,
    dummy_val_pred
)

print("Dummy validation accuracy:", dummy_val_accuracy)

Dummy validation accuracy: 0.5343283582089552


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))   
])

log_reg.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['return_1d','return_3d','return_5d',...,'overnight_gap','intraday_return', 'volume_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [14]:
log_val_pred = log_reg.predict(X_val)

log_val_prob = log_reg.predict_proba(X_val)[:,1]

log_val_accuracy = accuracy_score(
    y_val,
    log_val_pred
)

log_val_auc = roc_auc_score(
    y_val,
    log_val_prob
)

print("Dummy accuracy:  ", dummy_val_accuracy)
print("LogReg accuracy: ", log_val_accuracy)
print("LogReg ROC-AUC:", log_val_auc)

Dummy accuracy:   0.5343283582089552
LogReg accuracy:  0.545273631840796
LogReg ROC-AUC: 0.5414060386127426


In [15]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)

tree.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at

In [16]:
tree_val_pred = tree.predict(X_val)

tree_val_prob = tree.predict_proba(X_val)[:,1]

tree_val_accuracy = accuracy_score(
    y_val,
    tree_val_pred
)

tree_val_auc = roc_auc_score(
    y_val,
    tree_val_prob
)

print("Dummy accuracy: ", dummy_val_accuracy)
print("LogReg accuracy:", log_val_accuracy)
print("LogReg AUC:     ", log_val_auc)
print()
print("Tree accuracy:  ", tree_val_accuracy)
print("Tree ROC-AUC:   ", tree_val_auc)

Dummy accuracy:  0.5343283582089552
LogReg accuracy: 0.545273631840796
LogReg AUC:      0.5414060386127426

Tree accuracy:   0.5492537313432836
Tree ROC-AUC:    0.5518013178627704


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=50,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [18]:
rf_val_pred = rf.predict(X_val)
rf_val_prob = rf.predict_proba(X_val)[:,1]

rf_val_accuracy = accuracy_score(
    y_val,
    rf_val_pred
)

rf_val_auc = roc_auc_score(
    y_val,
    rf_val_prob
)

print("Random Forest accuracy:", rf_val_accuracy)
print("Random Forest ROC-AUC: ", rf_val_auc)


print("Dummy accuracy:", dummy_val_accuracy)

print("\nLogReg")
print("Accuracy:", log_val_accuracy)
print("AUC:     ", log_val_auc)

print("\nDecision Tree")
print("Accuracy:", tree_val_accuracy)
print("AUC:     ", tree_val_auc)

print("\nRandom Forest")
print("Accuracy:", rf_val_accuracy)
print("AUC:     ", rf_val_auc)

Random Forest accuracy: 0.5383084577114428
Random Forest ROC-AUC:  0.5521136736220534
Dummy accuracy: 0.5343283582089552

LogReg
Accuracy: 0.545273631840796
AUC:      0.5414060386127426

Decision Tree
Accuracy: 0.5492537313432836
AUC:      0.5518013178627704

Random Forest
Accuracy: 0.5383084577114428
AUC:      0.5521136736220534


In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    learning_rate = 0.05,
    max_iter = 200,
    max_leaf_nodes = 15,
    min_samples_leaf =50,
    random_state= 42
)

hgb.fit(X_train,y_train)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",15
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",50
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing va

In [20]:
hgb_val_pred = hgb.predict(X_val)
hgb_val_prob = hgb.predict_proba(X_val)[:,1]

hgb_val_accuracy = accuracy_score(
    y_val,
    hgb_val_pred
)

hgb_val_auc = roc_auc_score(
    y_val,
    hgb_val_prob
)

print("Gradient Boosting accuracy:", hgb_val_accuracy)
print("Gradient Boosting ROC-AUC: ", hgb_val_auc)

Gradient Boosting accuracy: 0.545273631840796
Gradient Boosting ROC-AUC:  0.5450269779878718


In [21]:
from xgboost import XGBClassifier

In [22]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [23]:
xgb_val_pred = xgb.predict(X_val)
xgb_val_prob = xgb.predict_proba(X_val)[:, 1]

xgb_val_accuracy = accuracy_score(
    y_val,
    xgb_val_pred
)

xgb_val_auc = roc_auc_score(
    y_val,
    xgb_val_prob
)

print("XGBoost accuracy:", xgb_val_accuracy)
print("XGBoost ROC-AUC: ", xgb_val_auc)

XGBoost accuracy: 0.5393034825870647
XGBoost ROC-AUC:  0.5462565057537125


In [24]:
results = pd.DataFrame({
    "Model": [
        "Dummy",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "HistGradientBoosting",
        "XGBoost"
    ],
    "Accuracy": [
        dummy_val_accuracy,
        log_val_accuracy,
        tree_val_accuracy,
        rf_val_accuracy,
        hgb_val_accuracy,
        xgb_val_accuracy
    ],
    "ROC_AUC": [
        np.nan,
        log_val_auc,
        tree_val_auc,
        rf_val_auc,
        hgb_val_auc,
        xgb_val_auc
    ]
})

results.sort_values("ROC_AUC", ascending=False)

,Model,Accuracy,ROC_AUC
3,Random Forest,0.538308,0.552114
2,Decision Tree,0.549254,0.551801
5,XGBoost,0.539303,0.546257
4,HistGradientBoosting,0.545274,0.545027
1,Logistic Regression,0.545274,0.541406
0,Dummy,0.534328,NaN
